In [2]:
import numpy as np
import pandas as pd
from pathlib import Path


# =========================================================
# SETTINGS
# =========================================================
ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

LAG_PREFIXES = ["lag5", "lag10", "lag21", "lag63"]
TARGET_PREFIX = "target"

INPUT_FILES = {
    10: "../proxy/realized_cov_target_h10_lag_5_10_21_63.csv",
    21: "../proxy/realized_cov_target_h21_lag_5_10_21_63.csv",
    63: "../proxy/realized_cov_target_h63_lag_5_10_21_63.csv",
}

OUTPUT_TEMPLATE = "../proxy/realized_chol_target_h{h}_lag_5_10_21_63.csv"

Grid search for SVR, here ranking on Frobenius 

In [21]:
import numpy as np
import pandas as pd
import optuna
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# =========================================================
# OPTUNA SEARCH FOR SVR-CHOLESKY COVARIANCE MODEL
# Rank by average Frobenius, keep QLIKE too
# Validation period: 2019-2020
# Compare against realized covariance proxy
# Refit every 21 forecast origins
# SPD is enforced in evaluation
# =========================================================

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------
TARGET_H = 10

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
PROXY_PATH = f"../proxy/realized_cov_h{TARGET_H}.csv"

ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

VALIDATION_START = "2019-01-01"
if TARGET_H == 10:
    VALIDATION_END = "2020-12-17"
elif TARGET_H == 21:
    VALIDATION_END = "2020-12-02"
elif TARGET_H == 63:
    VALIDATION_END = "2020-10-02"
else:
    raise ValueError("TARGET_H must be one of {10, 21, 63}")

LAGS = (10, 21, 63)
REFIT_EVERY = 1

RIDGE_EPS = 1e-8
RANDOM_STATE = 42
N_TRIALS = 50

SAVE_TRIALS_PATH = f"../results/svr_optuna_trials_h{TARGET_H}.csv"
SAVE_BEST_PATH = f"../results/svr_optuna_best_h{TARGET_H}.csv"

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
chol = (
    pd.read_csv(CHOL_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .reset_index(drop=True)
)

proxy = (
    pd.read_csv(PROXY_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .set_index("Date")
)

validation_dates = chol.loc[
    (chol["Date"] >= pd.Timestamp(VALIDATION_START)) &
    (chol["Date"] <= pd.Timestamp(VALIDATION_END)),
    "Date"
]

print("Loaded Cholesky data:", CHOL_PATH)
print("Loaded proxy data:", PROXY_PATH)
print("Validation:", validation_dates.min(), "->", validation_dates.max())
print("Refit every:", REFIT_EVERY)
print("Number of Optuna trials:", N_TRIALS)
print()

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs

def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]

def vec_to_matrix(vec, n_assets):
    M = np.asarray(vec, dtype=float).reshape(n_assets, n_assets)
    M = 0.5 * (M + M.T)
    return M

def frobenius_distance(S_true, H_pred):
    return float(np.linalg.norm(S_true - H_pred, ord="fro"))

def make_spd(matrix, min_eig=1e-8, jitter=1e-12):
    matrix = np.asarray(matrix, dtype=float)
    matrix = 0.5 * (matrix + matrix.T)

    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.clip(eigvals, min_eig, None)

    spd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    spd = 0.5 * (spd + spd.T)
    spd += jitter * np.eye(spd.shape[0])

    return spd


def qlike_loss(S_true, H_pred, eps=RIDGE_EPS):
    """
    Matrix QLIKE:
        tr(H_pred^{-1} S_true) - logdet(S_true) + logdet(H_pred) - n
    """
    S_true = make_spd(S_true, min_eig=eps)
    H_pred = make_spd(H_pred, min_eig=eps)

    n = S_true.shape[0]

    sign_s, logdet_s = np.linalg.slogdet(S_true)
    sign_h, logdet_h = np.linalg.slogdet(H_pred)

    if sign_s <= 0 or sign_h <= 0:
        return np.nan

    try:
        trace_term = np.trace(np.linalg.solve(H_pred, S_true))
    except np.linalg.LinAlgError:
        return np.nan

    return float(trace_term - logdet_s + logdet_h - n)

lower_pairs = build_lower_tri_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

proxy_cols = [f"cov_{a}__{b}" for a, b in full_pairs]

n_assets = len(ASSET_ORDER)
asset_to_idx = {a: i for i, a in enumerate(ASSET_ORDER)}

missing_proxy_cols = [c for c in proxy_cols if c not in proxy.columns]
if missing_proxy_cols:
    raise ValueError(f"Missing proxy columns, e.g. {missing_proxy_cols[:5]}")

date_to_loc = {d: i for i, d in enumerate(chol["Date"])}

# ---------------------------------------------------------
# MODEL EVALUATION FUNCTION
# ---------------------------------------------------------
def evaluate_svr_spec(window, C, gamma, epsilon):
    models = None
    scalers_x = None
    scalers_y = None
    last_refit_loc = None

    frob_list = []
    qlike_list = []
    n_forecasts = 0
    n_refits = 0

    for p in validation_dates:
        p_loc = date_to_loc[p]

        should_refit = (
            models is None
            or last_refit_loc is None
            or (p_loc - last_refit_loc) >= REFIT_EVERY
        )

        if should_refit:
            train_end = p_loc - TARGET_H
            train_start = train_end - window + 1

            if train_start < 0:
                continue

            train = chol.iloc[train_start:train_end + 1]

            models = {}
            scalers_x = {}
            scalers_y = {}

            for a, b in lower_pairs:
                target = f"target_chol_{a}__{b}"
                lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

                missing_cols = [c for c in lag_cols + [target] if c not in train.columns]
                if missing_cols:
                    raise ValueError(f"Missing columns for {(a, b)}: {missing_cols}")

                X = train[lag_cols].values
                y = train[target].values.reshape(-1, 1)

                scaler_x = StandardScaler()
                scaler_y = StandardScaler()

                Xs = scaler_x.fit_transform(X)
                ys = scaler_y.fit_transform(y).ravel()

                model = SVR(
                    kernel="rbf",
                    C=C,
                    gamma=gamma,
                    epsilon=epsilon
                )
                model.fit(Xs, ys)

                models[(a, b)] = model
                scalers_x[(a, b)] = scaler_x
                scalers_y[(a, b)] = scaler_y

            last_refit_loc = p_loc
            n_refits += 1

        if models is None:
            continue

        row = chol.iloc[p_loc]
        L = np.zeros((n_assets, n_assets), dtype=float)

        for a, b in lower_pairs:
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
            X_now = row[lag_cols].values.reshape(1, -1)

            scaler_x = scalers_x[(a, b)]
            scaler_y = scalers_y[(a, b)]
            model = models[(a, b)]

            Xs_now = scaler_x.transform(X_now)
            y_scaled = model.predict(Xs_now)

            y_pred = scaler_y.inverse_transform(y_scaled.reshape(-1, 1))[0, 0]

            i = asset_to_idx[a]
            j = asset_to_idx[b]
            L[i, j] = y_pred

        Sigma = L @ L.T
        Sigma = make_spd(Sigma, min_eig=1e-8, jitter=1e-12)

        if p not in proxy.index:
            continue

        true_vec = proxy.loc[p, proxy_cols].values
        S_true = vec_to_matrix(true_vec, n_assets)
        S_true = make_spd(S_true, min_eig=RIDGE_EPS)

        frob = frobenius_distance(S_true, Sigma)
        qlike = qlike_loss(S_true, Sigma, eps=RIDGE_EPS)

        if np.isfinite(frob) and np.isfinite(qlike):
            frob_list.append(frob)
            qlike_list.append(qlike)
            n_forecasts += 1

    return {
        "window": window,
        "C": C,
        "gamma": gamma,
        "epsilon": epsilon,
        "avg_qlike": np.mean(qlike_list) if len(qlike_list) > 0 else np.nan,
        "avg_frobenius": np.mean(frob_list) if len(frob_list) > 0 else np.nan,
        "n_forecasts": n_forecasts,
        "n_refits": n_refits
    }

# ---------------------------------------------------------
# OPTUNA OBJECTIVE
# ---------------------------------------------------------
trial_results = []

def objective(trial):
    window = trial.suggest_categorical(
        "window",
        [126, 189, 252, 315, 378, 441, 504]
    )

    C = trial.suggest_float("C", 1e-3, 1e3, log=True)

    gamma_mode = trial.suggest_categorical("gamma_mode", ["scale", "numeric"])
    if gamma_mode == "scale":
        gamma = "scale"
    else:
        gamma = trial.suggest_float("gamma", 1e-4, 1e1, log=True)

    epsilon = trial.suggest_float("epsilon", 1e-5, 1e-1, log=True)

    res = evaluate_svr_spec(
        window=window,
        C=C,
        gamma=gamma,
        epsilon=epsilon
    )

    trial.set_user_attr("avg_frobenius", res["avg_frobenius"])
    trial.set_user_attr("avg_qlike", res["avg_qlike"])
    trial.set_user_attr("n_forecasts", res["n_forecasts"])
    trial.set_user_attr("n_refits", res["n_refits"])
    trial.set_user_attr("gamma_used", gamma)

    trial_results.append(res)

    if not np.isfinite(res["avg_frobenius"]):
        return 1e12

    return res["avg_frobenius"]

# ---------------------------------------------------------
# RUN OPTUNA
# ---------------------------------------------------------
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# ---------------------------------------------------------
# COLLECT RESULTS
# ---------------------------------------------------------
trials_df = study.trials_dataframe().copy()

trials_df["avg_frobenius"] = [t.user_attrs.get("avg_frobenius", np.nan) for t in study.trials]
trials_df["avg_qlike"] = [t.user_attrs.get("avg_qlike", np.nan) for t in study.trials]
trials_df["n_forecasts"] = [t.user_attrs.get("n_forecasts", np.nan) for t in study.trials]
trials_df["n_refits"] = [t.user_attrs.get("n_refits", np.nan) for t in study.trials]
trials_df["gamma_used"] = [t.user_attrs.get("gamma_used", np.nan) for t in study.trials]

if "value" in trials_df.columns:
    trials_df = trials_df.rename(columns={"value": "objective_value"})

rename_map = {
    "params_window": "window",
    "params_C": "C",
    "params_gamma_mode": "gamma_mode",
    "params_gamma": "gamma",
    "params_epsilon": "epsilon",
}
trials_df = trials_df.rename(columns=rename_map)

keep_cols = [
    "number",
    "state",
    "avg_frobenius",
    "avg_qlike",
    "objective_value",
    "window",
    "C",
    "gamma_mode",
    "gamma",
    "gamma_used",
    "epsilon",
    "n_forecasts",
    "n_refits",
    "datetime_start",
    "datetime_complete",
    "duration",
]
keep_cols = [c for c in keep_cols if c in trials_df.columns]
trials_df = trials_df[keep_cols]

trials_df = trials_df.sort_values(
    ["avg_frobenius", "avg_qlike", "window"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# SAVE + PRINT
# ---------------------------------------------------------
trials_df.to_csv(SAVE_TRIALS_PATH, index=False)

best_params_df = pd.DataFrame([study.best_params])
best_params_df["best_avg_frobenius"] = study.best_value
best_params_df["best_avg_qlike"] = (
    trials_df.iloc[0]["avg_qlike"] if len(trials_df) > 0 else np.nan
)

best_gamma_mode = study.best_params.get("gamma_mode")
if best_gamma_mode == "scale":
    best_gamma_used = "scale"
else:
    best_gamma_used = study.best_params.get("gamma")

best_params_df["gamma_used"] = best_gamma_used
best_params_df.to_csv(SAVE_BEST_PATH, index=False)

print("\n=========================================================")
print("OPTUNA SVR SEARCH COMPLETE")
print("Forecast horizon:", TARGET_H)
print(f"Validation period: {VALIDATION_START} -> {VALIDATION_END}")
print(f"Refit every: {REFIT_EVERY} forecast origins")
print("=========================================================\n")

print("Top trials ranked by Frobenius:\n")
print(trials_df.head(20).to_string(index=False))

print("\n=========================================================")
print("BEST OPTUNA MODEL")
print("=========================================================")
print("Best avg Frobenius:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("  gamma_used:", best_gamma_used)

print("\nSaved trial results to:", SAVE_TRIALS_PATH)
print("Saved best parameters to:", SAVE_BEST_PATH)

[I 2026-04-15 09:32:48,197] A new study created in memory with name: no-name-155cc06d-aef6-416a-a789-79f9a13420b4


Loaded Cholesky data: ../proxy/realized_chol_target_h10_lag_5_10_21_63.csv
Loaded proxy data: ../proxy/realized_cov_h10.csv
Validation: 2019-01-02 00:00:00 -> 2020-12-17 00:00:00
Refit every: 1
Number of Optuna trials: 50



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-15 09:33:49,196] Trial 0 finished with value: 0.00250011077651201 and parameters: {'window': 189, 'C': 157.41890047456639, 'gamma_mode': 'numeric', 'gamma': 0.00012674255898937226, 'epsilon': 0.07579479953347998}. Best is trial 0 with value: 0.00250011077651201.
[I 2026-04-15 09:34:26,206] Trial 1 finished with value: 0.0023965975686278066 and parameters: {'window': 126, 'C': 0.055895242052179224, 'gamma_mode': 'scale', 'epsilon': 0.00014742753159914678}. Best is trial 1 with value: 0.0023965975686278066.
[I 2026-04-15 09:35:57,698] Trial 2 finished with value: 0.0026633011099767736 and parameters: {'window': 252, 'C': 4.418441521199721, 'gamma_mode': 'scale', 'epsilon': 0.062451395747430666}. Best is trial 1 with value: 0.0023965975686278066.
[I 2026-04-15 09:36:33,187] Trial 3 finished with value: 0.00240291667410429 and parameters: {'window': 126, 'C': 0.9355380606452183, 'gamma_mode': 'numeric', 'gamma': 0.0019674328025306126, 'epsilon': 0.004467752817973905}. Best is tr

In [22]:
import numpy as np
import pandas as pd
import optuna
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# =========================================================
# OPTUNA SEARCH FOR SVR-CHOLESKY COVARIANCE MODEL
# Rank by average Frobenius, keep QLIKE too
# Validation period: 2019-2020
# Compare against realized covariance proxy
# Refit every 21 forecast origins
# SPD is enforced in evaluation
# =========================================================

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------
TARGET_H = 21

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
PROXY_PATH = f"../proxy/realized_cov_h{TARGET_H}.csv"

ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

VALIDATION_START = "2019-01-01"
if TARGET_H == 10:
    VALIDATION_END = "2020-12-17"
elif TARGET_H == 21:
    VALIDATION_END = "2020-12-02"
elif TARGET_H == 63:
    VALIDATION_END = "2020-10-02"
else:
    raise ValueError("TARGET_H must be one of {10, 21, 63}")

LAGS = (10, 21, 63)
REFIT_EVERY = 1

RIDGE_EPS = 1e-8
RANDOM_STATE = 42
N_TRIALS = 50

SAVE_TRIALS_PATH = f"../results/svr_optuna_trials_h{TARGET_H}.csv"
SAVE_BEST_PATH = f"../results/svr_optuna_best_h{TARGET_H}.csv"

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
chol = (
    pd.read_csv(CHOL_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .reset_index(drop=True)
)

proxy = (
    pd.read_csv(PROXY_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .set_index("Date")
)

validation_dates = chol.loc[
    (chol["Date"] >= pd.Timestamp(VALIDATION_START)) &
    (chol["Date"] <= pd.Timestamp(VALIDATION_END)),
    "Date"
]

print("Loaded Cholesky data:", CHOL_PATH)
print("Loaded proxy data:", PROXY_PATH)
print("Validation:", validation_dates.min(), "->", validation_dates.max())
print("Refit every:", REFIT_EVERY)
print("Number of Optuna trials:", N_TRIALS)
print()

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs

def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]

def vec_to_matrix(vec, n_assets):
    M = np.asarray(vec, dtype=float).reshape(n_assets, n_assets)
    M = 0.5 * (M + M.T)
    return M

def frobenius_distance(S_true, H_pred):
    return float(np.linalg.norm(S_true - H_pred, ord="fro"))

def make_spd(matrix, min_eig=1e-8, jitter=1e-12):
    matrix = np.asarray(matrix, dtype=float)
    matrix = 0.5 * (matrix + matrix.T)

    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.clip(eigvals, min_eig, None)

    spd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    spd = 0.5 * (spd + spd.T)
    spd += jitter * np.eye(spd.shape[0])

    return spd


def qlike_loss(S_true, H_pred, eps=RIDGE_EPS):
    """
    Matrix QLIKE:
        tr(H_pred^{-1} S_true) - logdet(S_true) + logdet(H_pred) - n
    """
    S_true = make_spd(S_true, min_eig=eps)
    H_pred = make_spd(H_pred, min_eig=eps)

    n = S_true.shape[0]

    sign_s, logdet_s = np.linalg.slogdet(S_true)
    sign_h, logdet_h = np.linalg.slogdet(H_pred)

    if sign_s <= 0 or sign_h <= 0:
        return np.nan

    try:
        trace_term = np.trace(np.linalg.solve(H_pred, S_true))
    except np.linalg.LinAlgError:
        return np.nan

    return float(trace_term - logdet_s + logdet_h - n)

lower_pairs = build_lower_tri_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

proxy_cols = [f"cov_{a}__{b}" for a, b in full_pairs]

n_assets = len(ASSET_ORDER)
asset_to_idx = {a: i for i, a in enumerate(ASSET_ORDER)}

missing_proxy_cols = [c for c in proxy_cols if c not in proxy.columns]
if missing_proxy_cols:
    raise ValueError(f"Missing proxy columns, e.g. {missing_proxy_cols[:5]}")

date_to_loc = {d: i for i, d in enumerate(chol["Date"])}

# ---------------------------------------------------------
# MODEL EVALUATION FUNCTION
# ---------------------------------------------------------
def evaluate_svr_spec(window, C, gamma, epsilon):
    models = None
    scalers_x = None
    scalers_y = None
    last_refit_loc = None

    frob_list = []
    qlike_list = []
    n_forecasts = 0
    n_refits = 0

    for p in validation_dates:
        p_loc = date_to_loc[p]

        should_refit = (
            models is None
            or last_refit_loc is None
            or (p_loc - last_refit_loc) >= REFIT_EVERY
        )

        if should_refit:
            train_end = p_loc - TARGET_H
            train_start = train_end - window + 1

            if train_start < 0:
                continue

            train = chol.iloc[train_start:train_end + 1]

            models = {}
            scalers_x = {}
            scalers_y = {}

            for a, b in lower_pairs:
                target = f"target_chol_{a}__{b}"
                lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

                missing_cols = [c for c in lag_cols + [target] if c not in train.columns]
                if missing_cols:
                    raise ValueError(f"Missing columns for {(a, b)}: {missing_cols}")

                X = train[lag_cols].values
                y = train[target].values.reshape(-1, 1)

                scaler_x = StandardScaler()
                scaler_y = StandardScaler()

                Xs = scaler_x.fit_transform(X)
                ys = scaler_y.fit_transform(y).ravel()

                model = SVR(
                    kernel="rbf",
                    C=C,
                    gamma=gamma,
                    epsilon=epsilon
                )
                model.fit(Xs, ys)

                models[(a, b)] = model
                scalers_x[(a, b)] = scaler_x
                scalers_y[(a, b)] = scaler_y

            last_refit_loc = p_loc
            n_refits += 1

        if models is None:
            continue

        row = chol.iloc[p_loc]
        L = np.zeros((n_assets, n_assets), dtype=float)

        for a, b in lower_pairs:
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
            X_now = row[lag_cols].values.reshape(1, -1)

            scaler_x = scalers_x[(a, b)]
            scaler_y = scalers_y[(a, b)]
            model = models[(a, b)]

            Xs_now = scaler_x.transform(X_now)
            y_scaled = model.predict(Xs_now)

            y_pred = scaler_y.inverse_transform(y_scaled.reshape(-1, 1))[0, 0]

            i = asset_to_idx[a]
            j = asset_to_idx[b]
            L[i, j] = y_pred

        Sigma = L @ L.T
        Sigma = make_spd(Sigma, min_eig=1e-8, jitter=1e-12)

        if p not in proxy.index:
            continue

        true_vec = proxy.loc[p, proxy_cols].values
        S_true = vec_to_matrix(true_vec, n_assets)
        S_true = make_spd(S_true, min_eig=RIDGE_EPS)

        frob = frobenius_distance(S_true, Sigma)
        qlike = qlike_loss(S_true, Sigma, eps=RIDGE_EPS)

        if np.isfinite(frob) and np.isfinite(qlike):
            frob_list.append(frob)
            qlike_list.append(qlike)
            n_forecasts += 1

    return {
        "window": window,
        "C": C,
        "gamma": gamma,
        "epsilon": epsilon,
        "avg_qlike": np.mean(qlike_list) if len(qlike_list) > 0 else np.nan,
        "avg_frobenius": np.mean(frob_list) if len(frob_list) > 0 else np.nan,
        "n_forecasts": n_forecasts,
        "n_refits": n_refits
    }

# ---------------------------------------------------------
# OPTUNA OBJECTIVE
# ---------------------------------------------------------
trial_results = []

def objective(trial):
    window = trial.suggest_categorical(
        "window",
        [126, 189, 252, 315, 378, 441, 504]
    )

    C = trial.suggest_float("C", 1e-3, 1e3, log=True)

    gamma_mode = trial.suggest_categorical("gamma_mode", ["scale", "numeric"])
    if gamma_mode == "scale":
        gamma = "scale"
    else:
        gamma = trial.suggest_float("gamma", 1e-4, 1e1, log=True)

    epsilon = trial.suggest_float("epsilon", 1e-5, 1e-1, log=True)

    res = evaluate_svr_spec(
        window=window,
        C=C,
        gamma=gamma,
        epsilon=epsilon
    )

    trial.set_user_attr("avg_frobenius", res["avg_frobenius"])
    trial.set_user_attr("avg_qlike", res["avg_qlike"])
    trial.set_user_attr("n_forecasts", res["n_forecasts"])
    trial.set_user_attr("n_refits", res["n_refits"])
    trial.set_user_attr("gamma_used", gamma)

    trial_results.append(res)

    if not np.isfinite(res["avg_frobenius"]):
        return 1e12

    return res["avg_frobenius"]

# ---------------------------------------------------------
# RUN OPTUNA
# ---------------------------------------------------------
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# ---------------------------------------------------------
# COLLECT RESULTS
# ---------------------------------------------------------
trials_df = study.trials_dataframe().copy()

trials_df["avg_frobenius"] = [t.user_attrs.get("avg_frobenius", np.nan) for t in study.trials]
trials_df["avg_qlike"] = [t.user_attrs.get("avg_qlike", np.nan) for t in study.trials]
trials_df["n_forecasts"] = [t.user_attrs.get("n_forecasts", np.nan) for t in study.trials]
trials_df["n_refits"] = [t.user_attrs.get("n_refits", np.nan) for t in study.trials]
trials_df["gamma_used"] = [t.user_attrs.get("gamma_used", np.nan) for t in study.trials]

if "value" in trials_df.columns:
    trials_df = trials_df.rename(columns={"value": "objective_value"})

rename_map = {
    "params_window": "window",
    "params_C": "C",
    "params_gamma_mode": "gamma_mode",
    "params_gamma": "gamma",
    "params_epsilon": "epsilon",
}
trials_df = trials_df.rename(columns=rename_map)

keep_cols = [
    "number",
    "state",
    "avg_frobenius",
    "avg_qlike",
    "objective_value",
    "window",
    "C",
    "gamma_mode",
    "gamma",
    "gamma_used",
    "epsilon",
    "n_forecasts",
    "n_refits",
    "datetime_start",
    "datetime_complete",
    "duration",
]
keep_cols = [c for c in keep_cols if c in trials_df.columns]
trials_df = trials_df[keep_cols]

trials_df = trials_df.sort_values(
    ["avg_frobenius", "avg_qlike", "window"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# SAVE + PRINT
# ---------------------------------------------------------
trials_df.to_csv(SAVE_TRIALS_PATH, index=False)

best_params_df = pd.DataFrame([study.best_params])
best_params_df["best_avg_frobenius"] = study.best_value
best_params_df["best_avg_qlike"] = (
    trials_df.iloc[0]["avg_qlike"] if len(trials_df) > 0 else np.nan
)

best_gamma_mode = study.best_params.get("gamma_mode")
if best_gamma_mode == "scale":
    best_gamma_used = "scale"
else:
    best_gamma_used = study.best_params.get("gamma")

best_params_df["gamma_used"] = best_gamma_used
best_params_df.to_csv(SAVE_BEST_PATH, index=False)

print("\n=========================================================")
print("OPTUNA SVR SEARCH COMPLETE")
print("Forecast horizon:", TARGET_H)
print(f"Validation period: {VALIDATION_START} -> {VALIDATION_END}")
print(f"Refit every: {REFIT_EVERY} forecast origins")
print("=========================================================\n")

print("Top trials ranked by Frobenius:\n")
print(trials_df.head(20).to_string(index=False))

print("\n=========================================================")
print("BEST OPTUNA MODEL")
print("=========================================================")
print("Best avg Frobenius:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("  gamma_used:", best_gamma_used)

print("\nSaved trial results to:", SAVE_TRIALS_PATH)
print("Saved best parameters to:", SAVE_BEST_PATH)

[I 2026-04-15 10:39:03,584] A new study created in memory with name: no-name-4f684b3a-f8b7-4720-819c-cd2411a4d436


Loaded Cholesky data: ../proxy/realized_chol_target_h21_lag_5_10_21_63.csv
Loaded proxy data: ../proxy/realized_cov_h21.csv
Validation: 2019-01-02 00:00:00 -> 2020-12-02 00:00:00
Refit every: 1
Number of Optuna trials: 50



  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-15 10:39:34,768] Trial 0 finished with value: 0.0021033024463799023 and parameters: {'window': 189, 'C': 157.41890047456639, 'gamma_mode': 'numeric', 'gamma': 0.00012674255898937226, 'epsilon': 0.07579479953347998}. Best is trial 0 with value: 0.0021033024463799023.
[I 2026-04-15 10:39:57,072] Trial 1 finished with value: 0.002104017051131361 and parameters: {'window': 126, 'C': 0.055895242052179224, 'gamma_mode': 'scale', 'epsilon': 0.00014742753159914678}. Best is trial 0 with value: 0.0021033024463799023.
[I 2026-04-15 10:40:53,237] Trial 2 finished with value: 0.00244681908033162 and parameters: {'window': 252, 'C': 4.418441521199721, 'gamma_mode': 'scale', 'epsilon': 0.062451395747430666}. Best is trial 0 with value: 0.0021033024463799023.
[I 2026-04-15 10:41:15,008] Trial 3 finished with value: 0.002082120953385081 and parameters: {'window': 126, 'C': 0.9355380606452183, 'gamma_mode': 'numeric', 'gamma': 0.0019674328025306126, 'epsilon': 0.004467752817973905}. Best is 

In [23]:
import numpy as np
import pandas as pd
import optuna
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

# =========================================================
# OPTUNA SEARCH FOR SVR-CHOLESKY COVARIANCE MODEL
# Rank by average Frobenius, keep QLIKE too
# Validation period: 2019-2020
# Compare against realized covariance proxy
# Refit every 21 forecast origins
# SPD is enforced in evaluation
# =========================================================

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------
TARGET_H = 63

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
PROXY_PATH = f"../proxy/realized_cov_h{TARGET_H}.csv"

ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

VALIDATION_START = "2019-01-01"
if TARGET_H == 10:
    VALIDATION_END = "2020-12-17"
elif TARGET_H == 21:
    VALIDATION_END = "2020-12-02"
elif TARGET_H == 63:
    VALIDATION_END = "2020-10-02"
else:
    raise ValueError("TARGET_H must be one of {10, 21, 63}")

LAGS = (10, 21, 63)
REFIT_EVERY = 1

RIDGE_EPS = 1e-8
RANDOM_STATE = 42
N_TRIALS = 50

SAVE_TRIALS_PATH = f"../results/svr_optuna_trials_h{TARGET_H}.csv"
SAVE_BEST_PATH = f"../results/svr_optuna_best_h{TARGET_H}.csv"

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
chol = (
    pd.read_csv(CHOL_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .reset_index(drop=True)
)

proxy = (
    pd.read_csv(PROXY_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .set_index("Date")
)

validation_dates = chol.loc[
    (chol["Date"] >= pd.Timestamp(VALIDATION_START)) &
    (chol["Date"] <= pd.Timestamp(VALIDATION_END)),
    "Date"
]

print("Loaded Cholesky data:", CHOL_PATH)
print("Loaded proxy data:", PROXY_PATH)
print("Validation:", validation_dates.min(), "->", validation_dates.max())
print("Refit every:", REFIT_EVERY)
print("Number of Optuna trials:", N_TRIALS)
print()

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs

def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]

def vec_to_matrix(vec, n_assets):
    M = np.asarray(vec, dtype=float).reshape(n_assets, n_assets)
    M = 0.5 * (M + M.T)
    return M

def frobenius_distance(S_true, H_pred):
    return float(np.linalg.norm(S_true - H_pred, ord="fro"))

def make_spd(matrix, min_eig=1e-8, jitter=1e-12):
    matrix = np.asarray(matrix, dtype=float)
    matrix = 0.5 * (matrix + matrix.T)

    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.clip(eigvals, min_eig, None)

    spd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    spd = 0.5 * (spd + spd.T)
    spd += jitter * np.eye(spd.shape[0])

    return spd


def qlike_loss(S_true, H_pred, eps=RIDGE_EPS):
    """
    Matrix QLIKE:
        tr(H_pred^{-1} S_true) - logdet(S_true) + logdet(H_pred) - n
    """
    S_true = make_spd(S_true, min_eig=eps)
    H_pred = make_spd(H_pred, min_eig=eps)

    n = S_true.shape[0]

    sign_s, logdet_s = np.linalg.slogdet(S_true)
    sign_h, logdet_h = np.linalg.slogdet(H_pred)

    if sign_s <= 0 or sign_h <= 0:
        return np.nan

    try:
        trace_term = np.trace(np.linalg.solve(H_pred, S_true))
    except np.linalg.LinAlgError:
        return np.nan

    return float(trace_term - logdet_s + logdet_h - n)

lower_pairs = build_lower_tri_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

proxy_cols = [f"cov_{a}__{b}" for a, b in full_pairs]

n_assets = len(ASSET_ORDER)
asset_to_idx = {a: i for i, a in enumerate(ASSET_ORDER)}

missing_proxy_cols = [c for c in proxy_cols if c not in proxy.columns]
if missing_proxy_cols:
    raise ValueError(f"Missing proxy columns, e.g. {missing_proxy_cols[:5]}")

date_to_loc = {d: i for i, d in enumerate(chol["Date"])}

# ---------------------------------------------------------
# MODEL EVALUATION FUNCTION
# ---------------------------------------------------------
def evaluate_svr_spec(window, C, gamma, epsilon):
    models = None
    scalers_x = None
    scalers_y = None
    last_refit_loc = None

    frob_list = []
    qlike_list = []
    n_forecasts = 0
    n_refits = 0

    for p in validation_dates:
        p_loc = date_to_loc[p]

        should_refit = (
            models is None
            or last_refit_loc is None
            or (p_loc - last_refit_loc) >= REFIT_EVERY
        )

        if should_refit:
            train_end = p_loc - TARGET_H
            train_start = train_end - window + 1

            if train_start < 0:
                continue

            train = chol.iloc[train_start:train_end + 1]

            models = {}
            scalers_x = {}
            scalers_y = {}

            for a, b in lower_pairs:
                target = f"target_chol_{a}__{b}"
                lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

                missing_cols = [c for c in lag_cols + [target] if c not in train.columns]
                if missing_cols:
                    raise ValueError(f"Missing columns for {(a, b)}: {missing_cols}")

                X = train[lag_cols].values
                y = train[target].values.reshape(-1, 1)

                scaler_x = StandardScaler()
                scaler_y = StandardScaler()

                Xs = scaler_x.fit_transform(X)
                ys = scaler_y.fit_transform(y).ravel()

                model = SVR(
                    kernel="rbf",
                    C=C,
                    gamma=gamma,
                    epsilon=epsilon
                )
                model.fit(Xs, ys)

                models[(a, b)] = model
                scalers_x[(a, b)] = scaler_x
                scalers_y[(a, b)] = scaler_y

            last_refit_loc = p_loc
            n_refits += 1

        if models is None:
            continue

        row = chol.iloc[p_loc]
        L = np.zeros((n_assets, n_assets), dtype=float)

        for a, b in lower_pairs:
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
            X_now = row[lag_cols].values.reshape(1, -1)

            scaler_x = scalers_x[(a, b)]
            scaler_y = scalers_y[(a, b)]
            model = models[(a, b)]

            Xs_now = scaler_x.transform(X_now)
            y_scaled = model.predict(Xs_now)

            y_pred = scaler_y.inverse_transform(y_scaled.reshape(-1, 1))[0, 0]

            i = asset_to_idx[a]
            j = asset_to_idx[b]
            L[i, j] = y_pred

        Sigma = L @ L.T
        Sigma = make_spd(Sigma, min_eig=1e-8, jitter=1e-12)

        if p not in proxy.index:
            continue

        true_vec = proxy.loc[p, proxy_cols].values
        S_true = vec_to_matrix(true_vec, n_assets)
        S_true = make_spd(S_true, min_eig=RIDGE_EPS)

        frob = frobenius_distance(S_true, Sigma)
        qlike = qlike_loss(S_true, Sigma, eps=RIDGE_EPS)

        if np.isfinite(frob) and np.isfinite(qlike):
            frob_list.append(frob)
            qlike_list.append(qlike)
            n_forecasts += 1

    return {
        "window": window,
        "C": C,
        "gamma": gamma,
        "epsilon": epsilon,
        "avg_qlike": np.mean(qlike_list) if len(qlike_list) > 0 else np.nan,
        "avg_frobenius": np.mean(frob_list) if len(frob_list) > 0 else np.nan,
        "n_forecasts": n_forecasts,
        "n_refits": n_refits
    }

# ---------------------------------------------------------
# OPTUNA OBJECTIVE
# ---------------------------------------------------------
trial_results = []

def objective(trial):
    window = trial.suggest_categorical(
        "window",
        [126, 189, 252, 315, 378, 441, 504]
    )

    C = trial.suggest_float("C", 1e-3, 1e3, log=True)

    gamma_mode = trial.suggest_categorical("gamma_mode", ["scale", "numeric"])
    if gamma_mode == "scale":
        gamma = "scale"
    else:
        gamma = trial.suggest_float("gamma", 1e-4, 1e1, log=True)

    epsilon = trial.suggest_float("epsilon", 1e-5, 1e-1, log=True)

    res = evaluate_svr_spec(
        window=window,
        C=C,
        gamma=gamma,
        epsilon=epsilon
    )

    trial.set_user_attr("avg_frobenius", res["avg_frobenius"])
    trial.set_user_attr("avg_qlike", res["avg_qlike"])
    trial.set_user_attr("n_forecasts", res["n_forecasts"])
    trial.set_user_attr("n_refits", res["n_refits"])
    trial.set_user_attr("gamma_used", gamma)

    trial_results.append(res)

    if not np.isfinite(res["avg_frobenius"]):
        return 1e12

    return res["avg_frobenius"]

# ---------------------------------------------------------
# RUN OPTUNA
# ---------------------------------------------------------
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# ---------------------------------------------------------
# COLLECT RESULTS
# ---------------------------------------------------------
trials_df = study.trials_dataframe().copy()

trials_df["avg_frobenius"] = [t.user_attrs.get("avg_frobenius", np.nan) for t in study.trials]
trials_df["avg_qlike"] = [t.user_attrs.get("avg_qlike", np.nan) for t in study.trials]
trials_df["n_forecasts"] = [t.user_attrs.get("n_forecasts", np.nan) for t in study.trials]
trials_df["n_refits"] = [t.user_attrs.get("n_refits", np.nan) for t in study.trials]
trials_df["gamma_used"] = [t.user_attrs.get("gamma_used", np.nan) for t in study.trials]

if "value" in trials_df.columns:
    trials_df = trials_df.rename(columns={"value": "objective_value"})

rename_map = {
    "params_window": "window",
    "params_C": "C",
    "params_gamma_mode": "gamma_mode",
    "params_gamma": "gamma",
    "params_epsilon": "epsilon",
}
trials_df = trials_df.rename(columns=rename_map)

keep_cols = [
    "number",
    "state",
    "avg_frobenius",
    "avg_qlike",
    "objective_value",
    "window",
    "C",
    "gamma_mode",
    "gamma",
    "gamma_used",
    "epsilon",
    "n_forecasts",
    "n_refits",
    "datetime_start",
    "datetime_complete",
    "duration",
]
keep_cols = [c for c in keep_cols if c in trials_df.columns]
trials_df = trials_df[keep_cols]

trials_df = trials_df.sort_values(
    ["avg_frobenius", "avg_qlike", "window"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# SAVE + PRINT
# ---------------------------------------------------------
trials_df.to_csv(SAVE_TRIALS_PATH, index=False)

best_params_df = pd.DataFrame([study.best_params])
best_params_df["best_avg_frobenius"] = study.best_value
best_params_df["best_avg_qlike"] = (
    trials_df.iloc[0]["avg_qlike"] if len(trials_df) > 0 else np.nan
)

best_gamma_mode = study.best_params.get("gamma_mode")
if best_gamma_mode == "scale":
    best_gamma_used = "scale"
else:
    best_gamma_used = study.best_params.get("gamma")

best_params_df["gamma_used"] = best_gamma_used
best_params_df.to_csv(SAVE_BEST_PATH, index=False)

print("\n=========================================================")
print("OPTUNA SVR SEARCH COMPLETE")
print("Forecast horizon:", TARGET_H)
print(f"Validation period: {VALIDATION_START} -> {VALIDATION_END}")
print(f"Refit every: {REFIT_EVERY} forecast origins")
print("=========================================================\n")

print("Top trials ranked by Frobenius:\n")
print(trials_df.head(20).to_string(index=False))

print("\n=========================================================")
print("BEST OPTUNA MODEL")
print("=========================================================")
print("Best avg Frobenius:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("  gamma_used:", best_gamma_used)

print("\nSaved trial results to:", SAVE_TRIALS_PATH)
print("Saved best parameters to:", SAVE_BEST_PATH)

Loaded Cholesky data: ../proxy/realized_chol_target_h63_lag_5_10_21_63.csv
Loaded proxy data: ../proxy/realized_cov_h63.csv
Validation: 2019-01-02 00:00:00 -> 2020-10-02 00:00:00
Refit every: 1
Number of Optuna trials: 50



[I 2026-04-15 11:54:01,737] A new study created in memory with name: no-name-22d3b6d2-a94b-406f-b910-48b738246e63


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-15 11:54:29,342] Trial 0 finished with value: 0.007418765919831888 and parameters: {'window': 189, 'C': 157.41890047456639, 'gamma_mode': 'numeric', 'gamma': 0.00012674255898937226, 'epsilon': 0.07579479953347998}. Best is trial 0 with value: 0.007418765919831888.
[I 2026-04-15 11:54:48,750] Trial 1 finished with value: 0.002776202529381168 and parameters: {'window': 126, 'C': 0.055895242052179224, 'gamma_mode': 'scale', 'epsilon': 0.00014742753159914678}. Best is trial 1 with value: 0.002776202529381168.
[I 2026-04-15 11:55:39,687] Trial 2 finished with value: 0.0026464993426724404 and parameters: {'window': 252, 'C': 4.418441521199721, 'gamma_mode': 'scale', 'epsilon': 0.062451395747430666}. Best is trial 2 with value: 0.0026464993426724404.
[I 2026-04-15 11:55:59,591] Trial 3 finished with value: 0.0028406223064754697 and parameters: {'window': 126, 'C': 0.9355380606452183, 'gamma_mode': 'numeric', 'gamma': 0.0019674328025306126, 'epsilon': 0.004467752817973905}. Best is 

In [24]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# =========================================================
# SETTINGS
# =========================================================
TARGET_H = 10

ASSET_ORDER = [
    "US_EQUITY","INTL_EQUITY","US_BONDS","INTL_BONDS",
    "US_REIT","INTL_REIT","GOLD","BTC"
]

LAGS = (10,21,63)

TEST_START = "2021-01-01"

TRAINING_WINDOW = 189
REFIT_EVERY = 1

KERNEL = "rbf"
C = 1.95
GAMMA = 0.00104
EPSILON = 0.0000208

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
SAVE_PATH = f"../results/svr_cov_forecast_h{TARGET_H}.csv"

RIDGE_EPS = 1e-8
SPD_JITTER = 1e-12

# =========================================================
# HELPERS
# =========================================================
def build_lower_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs


def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]


def reconstruct_covariance(L):
    return L @ L.T


def make_spd(matrix, min_eig=RIDGE_EPS, jitter=SPD_JITTER):
    matrix = np.asarray(matrix, dtype=float)

    # Symmetrize
    matrix = 0.5 * (matrix + matrix.T)

    # Eigenvalue clipping
    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.clip(eigvals, min_eig, None)

    spd = eigvecs @ np.diag(eigvals) @ eigvecs.T

    # Final symmetrization + jitter
    spd = 0.5 * (spd + spd.T)
    spd += jitter * np.eye(spd.shape[0])

    return spd


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(CHOL_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

lower_pairs = build_lower_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

test_start = pd.Timestamp(TEST_START)
candidate_dates = df.loc[df["Date"] >= test_start, "Date"]

# =========================================================
# ROLLING FORECAST
# =========================================================
forecast_rows = []

models = None
scalers_x = None
scalers_y = None
last_refit_index = None

for p in candidate_dates:

    p_idx = df.index[df["Date"] == p][0]

    should_refit = (
        models is None
        or last_refit_index is None
        or (p_idx - last_refit_index) >= REFIT_EVERY
    )

    if should_refit:

        train_end = p_idx - TARGET_H
        train_start = train_end - TRAINING_WINDOW + 1

        if train_start < 0:
            continue

        train = df.iloc[train_start:train_end+1]

        models = {}
        scalers_x = {}
        scalers_y = {}

        for a, b in lower_pairs:

            target_col = f"target_chol_{a}__{b}"
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

            X = train[lag_cols].values
            y = train[target_col].values.reshape(-1, 1)

            scaler_x = StandardScaler()
            scaler_y = StandardScaler()

            Xs = scaler_x.fit_transform(X)
            ys = scaler_y.fit_transform(y).ravel()

            model = SVR(
                kernel=KERNEL,
                C=C,
                epsilon=EPSILON,
                gamma=GAMMA
            )

            model.fit(Xs, ys)

            models[(a, b)] = model
            scalers_x[(a, b)] = scaler_x
            scalers_y[(a, b)] = scaler_y

        last_refit_index = p_idx

        print(
            f"Refit at {p.date()} | "
            f"train rows {train.index.min()} -> {train.index.max()}"
        )

    # =====================================================
    # PREDICT CHOLESKY FACTOR
    # =====================================================
    n = len(ASSET_ORDER)
    L_pred = np.zeros((n, n))

    row = df.iloc[p_idx]

    for a, b in lower_pairs:

        lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
        X_pred = row[lag_cols].values.reshape(1, -1)

        scaler_x = scalers_x[(a, b)]
        scaler_y = scalers_y[(a, b)]
        model = models[(a, b)]

        Xs = scaler_x.transform(X_pred)
        y_scaled = model.predict(Xs)

        y_pred = scaler_y.inverse_transform(
            y_scaled.reshape(-1, 1)
        )[0, 0]

        i = ASSET_ORDER.index(a)
        j = ASSET_ORDER.index(b)
        L_pred[i, j] = y_pred

    # =====================================================
    # RECONSTRUCT COVARIANCE
    # =====================================================
    Sigma = reconstruct_covariance(L_pred)
    Sigma = make_spd(Sigma, min_eig=RIDGE_EPS, jitter=SPD_JITTER)

    out_row = {"Date": p}

    for i, a in enumerate(ASSET_ORDER):
        for j, b in enumerate(ASSET_ORDER):
            out_row[f"cov_{a}__{b}"] = Sigma[i, j]

    forecast_rows.append(out_row)

# =========================================================
# SAVE FORECASTS
# =========================================================
forecast_df = pd.DataFrame(forecast_rows)

cols = ["Date"] + [
    f"cov_{a}__{b}" for a, b in full_pairs
]

forecast_df = forecast_df[cols]

Path("../results").mkdir(exist_ok=True)

forecast_df.to_csv(SAVE_PATH, index=False)

print()
print("Saved forecasts to:", SAVE_PATH)
print("Shape:", forecast_df.shape)

Refit at 2021-01-04 | train rows 688 -> 876
Refit at 2021-01-05 | train rows 689 -> 877
Refit at 2021-01-06 | train rows 690 -> 878
Refit at 2021-01-07 | train rows 691 -> 879
Refit at 2021-01-08 | train rows 692 -> 880
Refit at 2021-01-11 | train rows 693 -> 881
Refit at 2021-01-12 | train rows 694 -> 882
Refit at 2021-01-13 | train rows 695 -> 883
Refit at 2021-01-14 | train rows 696 -> 884
Refit at 2021-01-15 | train rows 697 -> 885
Refit at 2021-01-19 | train rows 698 -> 886
Refit at 2021-01-20 | train rows 699 -> 887
Refit at 2021-01-21 | train rows 700 -> 888
Refit at 2021-01-22 | train rows 701 -> 889
Refit at 2021-01-25 | train rows 702 -> 890
Refit at 2021-01-26 | train rows 703 -> 891
Refit at 2021-01-27 | train rows 704 -> 892
Refit at 2021-01-28 | train rows 705 -> 893
Refit at 2021-01-29 | train rows 706 -> 894
Refit at 2021-02-01 | train rows 707 -> 895
Refit at 2021-02-02 | train rows 708 -> 896
Refit at 2021-02-03 | train rows 709 -> 897
Refit at 2021-02-04 | train rows

In [25]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# =========================================================
# SETTINGS
# =========================================================
TARGET_H = 21

ASSET_ORDER = [
    "US_EQUITY","INTL_EQUITY","US_BONDS","INTL_BONDS",
    "US_REIT","INTL_REIT","GOLD","BTC"
]

LAGS = (10,21,63)

TEST_START = "2021-01-01"

TRAINING_WINDOW = 189
REFIT_EVERY = 1

KERNEL = "rbf"
TRAINING_WINDOW = 189
C = 0.0556
GAMMA = "scale"
EPSILON = 0.000127

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
SAVE_PATH = f"../results/svr_cov_forecast_h{TARGET_H}.csv"

RIDGE_EPS = 1e-8
SPD_JITTER = 1e-12

# =========================================================
# HELPERS
# =========================================================
def build_lower_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs


def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]


def reconstruct_covariance(L):
    return L @ L.T


def make_spd(matrix, min_eig=RIDGE_EPS, jitter=SPD_JITTER):
    matrix = np.asarray(matrix, dtype=float)

    # Symmetrize
    matrix = 0.5 * (matrix + matrix.T)

    # Eigenvalue clipping
    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.clip(eigvals, min_eig, None)

    spd = eigvecs @ np.diag(eigvals) @ eigvecs.T

    # Final symmetrization + jitter
    spd = 0.5 * (spd + spd.T)
    spd += jitter * np.eye(spd.shape[0])

    return spd


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(CHOL_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

lower_pairs = build_lower_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

test_start = pd.Timestamp(TEST_START)
candidate_dates = df.loc[df["Date"] >= test_start, "Date"]

# =========================================================
# ROLLING FORECAST
# =========================================================
forecast_rows = []

models = None
scalers_x = None
scalers_y = None
last_refit_index = None

for p in candidate_dates:

    p_idx = df.index[df["Date"] == p][0]

    should_refit = (
        models is None
        or last_refit_index is None
        or (p_idx - last_refit_index) >= REFIT_EVERY
    )

    if should_refit:

        train_end = p_idx - TARGET_H
        train_start = train_end - TRAINING_WINDOW + 1

        if train_start < 0:
            continue

        train = df.iloc[train_start:train_end+1]

        models = {}
        scalers_x = {}
        scalers_y = {}

        for a, b in lower_pairs:

            target_col = f"target_chol_{a}__{b}"
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

            X = train[lag_cols].values
            y = train[target_col].values.reshape(-1, 1)

            scaler_x = StandardScaler()
            scaler_y = StandardScaler()

            Xs = scaler_x.fit_transform(X)
            ys = scaler_y.fit_transform(y).ravel()

            model = SVR(
                kernel=KERNEL,
                C=C,
                epsilon=EPSILON,
                gamma=GAMMA
            )

            model.fit(Xs, ys)

            models[(a, b)] = model
            scalers_x[(a, b)] = scaler_x
            scalers_y[(a, b)] = scaler_y

        last_refit_index = p_idx

        print(
            f"Refit at {p.date()} | "
            f"train rows {train.index.min()} -> {train.index.max()}"
        )

    # =====================================================
    # PREDICT CHOLESKY FACTOR
    # =====================================================
    n = len(ASSET_ORDER)
    L_pred = np.zeros((n, n))

    row = df.iloc[p_idx]

    for a, b in lower_pairs:

        lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
        X_pred = row[lag_cols].values.reshape(1, -1)

        scaler_x = scalers_x[(a, b)]
        scaler_y = scalers_y[(a, b)]
        model = models[(a, b)]

        Xs = scaler_x.transform(X_pred)
        y_scaled = model.predict(Xs)

        y_pred = scaler_y.inverse_transform(
            y_scaled.reshape(-1, 1)
        )[0, 0]

        i = ASSET_ORDER.index(a)
        j = ASSET_ORDER.index(b)
        L_pred[i, j] = y_pred

    # =====================================================
    # RECONSTRUCT COVARIANCE
    # =====================================================
    Sigma = reconstruct_covariance(L_pred)
    Sigma = make_spd(Sigma, min_eig=RIDGE_EPS, jitter=SPD_JITTER)

    out_row = {"Date": p}

    for i, a in enumerate(ASSET_ORDER):
        for j, b in enumerate(ASSET_ORDER):
            out_row[f"cov_{a}__{b}"] = Sigma[i, j]

    forecast_rows.append(out_row)

# =========================================================
# SAVE FORECASTS
# =========================================================
forecast_df = pd.DataFrame(forecast_rows)

cols = ["Date"] + [
    f"cov_{a}__{b}" for a, b in full_pairs
]

forecast_df = forecast_df[cols]

Path("../results").mkdir(exist_ok=True)

forecast_df.to_csv(SAVE_PATH, index=False)

print()
print("Saved forecasts to:", SAVE_PATH)
print("Shape:", forecast_df.shape)

Refit at 2021-01-04 | train rows 677 -> 865
Refit at 2021-01-05 | train rows 678 -> 866
Refit at 2021-01-06 | train rows 679 -> 867
Refit at 2021-01-07 | train rows 680 -> 868
Refit at 2021-01-08 | train rows 681 -> 869
Refit at 2021-01-11 | train rows 682 -> 870
Refit at 2021-01-12 | train rows 683 -> 871
Refit at 2021-01-13 | train rows 684 -> 872
Refit at 2021-01-14 | train rows 685 -> 873
Refit at 2021-01-15 | train rows 686 -> 874
Refit at 2021-01-19 | train rows 687 -> 875
Refit at 2021-01-20 | train rows 688 -> 876
Refit at 2021-01-21 | train rows 689 -> 877
Refit at 2021-01-22 | train rows 690 -> 878
Refit at 2021-01-25 | train rows 691 -> 879
Refit at 2021-01-26 | train rows 692 -> 880
Refit at 2021-01-27 | train rows 693 -> 881
Refit at 2021-01-28 | train rows 694 -> 882
Refit at 2021-01-29 | train rows 695 -> 883
Refit at 2021-02-01 | train rows 696 -> 884
Refit at 2021-02-02 | train rows 697 -> 885
Refit at 2021-02-03 | train rows 698 -> 886
Refit at 2021-02-04 | train rows

In [26]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# =========================================================
# SETTINGS
# =========================================================
TARGET_H = 63

ASSET_ORDER = [
    "US_EQUITY","INTL_EQUITY","US_BONDS","INTL_BONDS",
    "US_REIT","INTL_REIT","GOLD","BTC"
]

LAGS = (10,21,63)

TEST_START = "2021-01-01"

TRAINING_WINDOW = 315
REFIT_EVERY = 1

KERNEL = "rbf"
C = 0.00646
GAMMA = "scale"
EPSILON = 0.000571

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
SAVE_PATH = f"../results/svr_cov_forecast_h{TARGET_H}.csv"

RIDGE_EPS = 1e-8
SPD_JITTER = 1e-12

# =========================================================
# HELPERS
# =========================================================
def build_lower_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs


def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]


def reconstruct_covariance(L):
    return L @ L.T


def make_spd(matrix, min_eig=RIDGE_EPS, jitter=SPD_JITTER):
    matrix = np.asarray(matrix, dtype=float)

    # Symmetrize
    matrix = 0.5 * (matrix + matrix.T)

    # Eigenvalue clipping
    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.clip(eigvals, min_eig, None)

    spd = eigvecs @ np.diag(eigvals) @ eigvecs.T

    # Final symmetrization + jitter
    spd = 0.5 * (spd + spd.T)
    spd += jitter * np.eye(spd.shape[0])

    return spd


# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(CHOL_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

lower_pairs = build_lower_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

test_start = pd.Timestamp(TEST_START)
candidate_dates = df.loc[df["Date"] >= test_start, "Date"]

# =========================================================
# ROLLING FORECAST
# =========================================================
forecast_rows = []

models = None
scalers_x = None
scalers_y = None
last_refit_index = None

for p in candidate_dates:

    p_idx = df.index[df["Date"] == p][0]

    should_refit = (
        models is None
        or last_refit_index is None
        or (p_idx - last_refit_index) >= REFIT_EVERY
    )

    if should_refit:

        train_end = p_idx - TARGET_H
        train_start = train_end - TRAINING_WINDOW + 1

        if train_start < 0:
            continue

        train = df.iloc[train_start:train_end+1]

        models = {}
        scalers_x = {}
        scalers_y = {}

        for a, b in lower_pairs:

            target_col = f"target_chol_{a}__{b}"
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

            X = train[lag_cols].values
            y = train[target_col].values.reshape(-1, 1)

            scaler_x = StandardScaler()
            scaler_y = StandardScaler()

            Xs = scaler_x.fit_transform(X)
            ys = scaler_y.fit_transform(y).ravel()

            model = SVR(
                kernel=KERNEL,
                C=C,
                epsilon=EPSILON,
                gamma=GAMMA
            )

            model.fit(Xs, ys)

            models[(a, b)] = model
            scalers_x[(a, b)] = scaler_x
            scalers_y[(a, b)] = scaler_y

        last_refit_index = p_idx

        print(
            f"Refit at {p.date()} | "
            f"train rows {train.index.min()} -> {train.index.max()}"
        )

    # =====================================================
    # PREDICT CHOLESKY FACTOR
    # =====================================================
    n = len(ASSET_ORDER)
    L_pred = np.zeros((n, n))

    row = df.iloc[p_idx]

    for a, b in lower_pairs:

        lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
        X_pred = row[lag_cols].values.reshape(1, -1)

        scaler_x = scalers_x[(a, b)]
        scaler_y = scalers_y[(a, b)]
        model = models[(a, b)]

        Xs = scaler_x.transform(X_pred)
        y_scaled = model.predict(Xs)

        y_pred = scaler_y.inverse_transform(
            y_scaled.reshape(-1, 1)
        )[0, 0]

        i = ASSET_ORDER.index(a)
        j = ASSET_ORDER.index(b)
        L_pred[i, j] = y_pred

    # =====================================================
    # RECONSTRUCT COVARIANCE
    # =====================================================
    Sigma = reconstruct_covariance(L_pred)
    Sigma = make_spd(Sigma, min_eig=RIDGE_EPS, jitter=SPD_JITTER)

    out_row = {"Date": p}

    for i, a in enumerate(ASSET_ORDER):
        for j, b in enumerate(ASSET_ORDER):
            out_row[f"cov_{a}__{b}"] = Sigma[i, j]

    forecast_rows.append(out_row)

# =========================================================
# SAVE FORECASTS
# =========================================================
forecast_df = pd.DataFrame(forecast_rows)

cols = ["Date"] + [
    f"cov_{a}__{b}" for a, b in full_pairs
]

forecast_df = forecast_df[cols]

Path("../results").mkdir(exist_ok=True)

forecast_df.to_csv(SAVE_PATH, index=False)

print()
print("Saved forecasts to:", SAVE_PATH)
print("Shape:", forecast_df.shape)

Refit at 2021-01-04 | train rows 509 -> 823
Refit at 2021-01-05 | train rows 510 -> 824
Refit at 2021-01-06 | train rows 511 -> 825
Refit at 2021-01-07 | train rows 512 -> 826
Refit at 2021-01-08 | train rows 513 -> 827
Refit at 2021-01-11 | train rows 514 -> 828
Refit at 2021-01-12 | train rows 515 -> 829
Refit at 2021-01-13 | train rows 516 -> 830
Refit at 2021-01-14 | train rows 517 -> 831
Refit at 2021-01-15 | train rows 518 -> 832
Refit at 2021-01-19 | train rows 519 -> 833
Refit at 2021-01-20 | train rows 520 -> 834
Refit at 2021-01-21 | train rows 521 -> 835
Refit at 2021-01-22 | train rows 522 -> 836
Refit at 2021-01-25 | train rows 523 -> 837
Refit at 2021-01-26 | train rows 524 -> 838
Refit at 2021-01-27 | train rows 525 -> 839
Refit at 2021-01-28 | train rows 526 -> 840
Refit at 2021-01-29 | train rows 527 -> 841
Refit at 2021-02-01 | train rows 528 -> 842
Refit at 2021-02-02 | train rows 529 -> 843
Refit at 2021-02-03 | train rows 530 -> 844
Refit at 2021-02-04 | train rows